# HAILA: treino da SLM (LoRA) no Colab

**Antes de tudo:** Ambiente de execução > Alterar o tipo de ambiente de execução > **T4 GPU** > Salvar.

Depois: **Ambiente de execução > Executar tudo**. Quando aparecer o botão *Escolher ficheiros*, envie os dois arquivos: o zip do projeto (`HAILACBQs-corrigido...zip`) e o `dados.zip`. O resto é automático (cerca de 40 a 60 minutos). Não clique nas células enquanto rodam.

In [ ]:
# 1. Confere a GPU (deve mostrar Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Envie o zip do projeto e o dados.zip
from google.colab import files
enviados = files.upload()
print('Recebidos:', list(enviados))

In [ ]:
# 3. Descompacta (acha os arquivos mesmo que o Colab tenha renomeado) e instala versões testadas
import glob, os
proj = max(glob.glob('/content/HAILACBQs-corrigido*.zip'), key=os.path.getmtime)
dados = max(glob.glob('/content/dados*.zip'), key=os.path.getmtime)
print('Projeto:', proj)
print('Dados:  ', dados)
!rm -rf /content/HAILACBQs-main
!unzip -q -o "$proj" -d /content
%cd /content/HAILACBQs-main
!rm -rf dados && unzip -q -o "$dados"
!ls dados
!pip install -q "transformers==4.46.3" "peft==0.13.2" "accelerate==1.1.1" 2>&1 | grep -v -i "dependency resolver\|requires huggingface" || true
print('\nPronto. Deve aparecer haila-lora-enade e haila-lora-enade-poscomp acima.')

### 4. Treino
O primeiro passo de cada treino leva de 30 s a 1 min para aparecer na barra (`0/…` parado no início é normal). Ordem: versão só ENADE (5–10 min), versão ENADE + POSCOMP (20–40 min), avaliação do Qwen base (alguns minutos), tabela comparativa.

In [ ]:
# 4. Treina as duas versões, avalia o Qwen base e compara
%cd /content/HAILACBQs-main
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python research/lora/treinar_tudo.py

In [ ]:
# 5. Resultado
print(open('/content/HAILACBQs-main/adapters/comparacao_validacao.md').read())

In [ ]:
# 6. Baixa os adapters treinados (adapters.zip)
%cd /content/HAILACBQs-main
!find adapters -name checkpoints -type d -prune -exec rm -rf {} +
!zip -qr /content/adapters.zip adapters
from google.colab import files
files.download('/content/adapters.zip')

**Na sua máquina:** descompacte `adapters.zip` dentro de `~/HAILA` e mande para a Claude o conteúdo de `adapters/comparacao_validacao.md`.